### **Check GPU**

In [2]:
import torch

if torch.cuda.is_available():
    print("GPU available: True")
    print("GPU name:", torch.cuda.get_device_name(0))
    print("VRAM:", round(
        torch.cuda.get_device_properties(0).total_memory / 1e9, 1
    ), "GB")
else:
    print("GPU not enabled.")

GPU available: True
GPU name: Tesla T4
VRAM: 15.6 GB


### **Install Unsloth**

In [3]:
!pip install unsloth -q
print("Installation complete")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.0/56.0 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.4/67.4 MB 12.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 12.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 45.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 94.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 37.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 428.0/428.0 kB 37.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 81.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 128.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 185.2/185.2 kB 21.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 18.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 121.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 225

### **Imports**

In [4]:
from unsloth import FastLanguageModel
from trl import SFTTrainer, SFTConfig
from datasets import Dataset
import json
import torch

print("All imports done")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
All imports done


### **Load and Verify Dataset**

In [8]:
# Load your healthcare dataset from uploaded file
with open("healthcare_data.json", "r") as f:
    raw_data = json.load(f)

print(f"Total examples loaded: {len(raw_data)}")
print(f"\nFirst example:")
print(f"Instruction: {raw_data[0]['instruction']}")
print(f"Output: {raw_data[0]['output'][:100]}...")
print(f"\nLast example:")
print(f"Instruction: {raw_data[-1]['instruction']}")

Total examples loaded: 336

First example:
Instruction: What are the common symptoms of type 2 diabetes?
Output: Common symptoms of type 2 diabetes include increased thirst, frequent urination, unexplained weight ...

Last example:
Instruction: Benefits of learning a new skill for brain health.


### **Format Dataset**

In [9]:
def format_example(example):

    # Build text in Qwen chat format
    text = (
        "<|im_start|>system\n"
        "You are a professional healthcare assistant. "
        "Provide accurate, clear, and helpful medical information. "
        "Always recommend consulting a doctor for personal medical advice."
        "<|im_end|>\n"
        "<|im_start|>user\n"
        f"{example['instruction']}<|im_end|>\n"
        "<|im_start|>assistant\n"
        f"{example['output']}<|im_end|>"
    )

    return {"text": text}

# Convert to HuggingFace Dataset
hf_dataset = Dataset.from_list(raw_data)

# Apply formatting
dataset = hf_dataset.map(
    format_example,
    remove_columns=hf_dataset.column_names
)

# Split 90% train 10% test
split      = dataset.train_test_split(test_size=0.1, seed=42)
train_data = split["train"]
test_data  = split["test"]

print(f"Train examples: {len(train_data)}")
print(f"Test examples:  {len(test_data)}")
print(f"\nFormatted sample:")
print(train_data[0]["text"][:400])

Map:   0%|          | 0/336 [00:00<?, ? examples/s]

Train examples: 302
Test examples:  34

Formatted sample:
<|im_start|>system
You are a professional healthcare assistant. Provide accurate, clear, and helpful medical information. Always recommend consulting a doctor for personal medical advice.<|im_end|>
<|im_start|>user
How does nature therapy (ecotherapy) work?<|im_end|>
<|im_start|>assistant
Nature therapy uses outdoor activities to reduce stress, improve mood, attention, and physical fitness.<|im_en


### **Load Qwen Model in 4-bit (QLoRA)**

In [10]:
MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = MODEL_NAME,
    max_seq_length = 1024,
    load_in_4bit = True,     # QLoRA — 4-bit base model
    dtype = None,
)

print(f"Model loaded: {MODEL_NAME}")
print(f"Parameters: {model.num_parameters():,}")

vram = torch.cuda.memory_allocated() / 1e9
print(f"VRAM used after loading: {vram:.2f} GB")

==((====))==  Unsloth 2026.5.2: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/538M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/270 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

...

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/605 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/614 [00:00<?, ?B/s]

unsloth/qwen2.5-0.5b-instruct-unsloth-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.
Model loaded: Qwen/Qwen2.5-0.5B-Instruct
Parameters: 494,032,768
VRAM used after loading: 0.57 GB


### **Add DoRA Adapters**

In [11]:
# Using DoRA — best quality adapter method
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,              # rank — higher than 135M because 0.5B is bigger
    lora_alpha = 32,     # 2 × r
    lora_dropout = 0.05,
    target_modules = [
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj"
    ],
    use_dora = True,     # DoRA — best quality
    random_state = 42,
)

model.print_trainable_parameters()

Unsloth: Dropout = 0 is supported for fast patching. You are using dropout = 0.05.
Unsloth will patch all other layers, except LoRA matrices, causing a performance hit.
Unsloth 2026.5.2 patched 24 layers with 0 QKV layers, 0 O layers and 0 MLP layers.


trainable params: 2,211,840 || all params: 496,244,608 || trainable%: 0.4457


### **Check VRAM Before Training**

In [12]:
vram_total = torch.cuda.get_device_properties(0).total_memory / 1e9
vram_used  = torch.cuda.memory_allocated() / 1e9
vram_free  = vram_total - vram_used

print(f"Total VRAM:  {vram_total:.1f} GB")
print(f"Used VRAM:   {vram_used:.2f} GB")
print(f"Free VRAM:   {vram_free:.2f} GB")

if vram_free > 5:
    print("VRAM status: Good to train")
elif vram_free > 2:
    print("VRAM status: Tight — reduce batch size if error")
else:
    print("VRAM status: Low — restart runtime")

Total VRAM:  15.6 GB
Used VRAM:   0.58 GB
Free VRAM:   15.06 GB
VRAM status: Good to train


### **Train**

In [13]:
trainer = SFTTrainer(
    model = model,
    train_dataset = train_data,
    eval_dataset = test_data,
    args = SFTConfig(
        output_dir = "./qwen-healthcare-dora",

        # Training
        num_train_epochs = 3,
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,

        # Learning rate
        learning_rate = 2e-4,
        warmup_steps = 10,           # ← fixed deprecation warning too
        lr_scheduler_type = "cosine",

        # Logging
        logging_steps = 10,
        eval_strategy = "steps",
        eval_steps = 50,
        save_steps = 100,
        save_total_limit = 2,

        # Precision — fp16 for T4 GPU
        fp16 = True,                 # ← changed from bf16 to fp16
        bf16 = False,                # ← explicitly off

        # Other
        report_to = "none",
        seed = 42,
    ),
)

print("=" * 50)
print("TRAINING: Qwen2.5-0.5B Healthcare Assistant")
print("Adapter:  DoRA")
print("Data:     500 healthcare examples")
print("Epochs:   3")
print("=" * 50)
print()

trainer.train()

print()
print("=" * 50)
print("TRAINING COMPLETE")
print("=" * 50)

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/605 [00:00<?, ?B/s]

...

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151654}.


🦥 Unsloth: Padding-free auto-enabled, enabling faster training.
TRAINING: Qwen2.5-0.5B Healthcare Assistant
Adapter:  DoRA
Data:     500 healthcare examples
Epochs:   3



==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 302 | Num Epochs = 3 | Total steps = 114
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 2,211,840 of 496,244,608 (0.45% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!


Step,Training Loss,Validation Loss
50,0.818500,0.819554
100,0.711277,0.785508
114,0.728228,0.785424



TRAINING COMPLETE


### **Test Healthcare Model**

In [14]:
FastLanguageModel.for_inference(model)

# Healthcare specific test prompts
test_prompts = [
    "What are the early symptoms of diabetes?",
    "What should I do if someone is having a heart attack?",
    "What are common side effects of ibuprofen?",
    "How many hours of sleep does an adult need?",
    "What are signs of depression I should watch for?",
]

print("=" * 55)
print("HEALTHCARE ASSISTANT — MODEL OUTPUT")
print("=" * 55)

for prompt in test_prompts:

    messages = [
        {
            "role": "system",
            "content": (
                "You are a professional healthcare assistant. "
                "Provide accurate, clear, and helpful medical information. "
                "Always recommend consulting a doctor for personal advice."
            )
        },
        {
            "role": "user",
            "content": prompt
        }
    ]

    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize = True,
        add_generation_prompt = True,
        return_tensors = "pt"
    ).to("cuda")

    outputs = model.generate(
        input_ids = inputs,
        max_new_tokens = 200,
        temperature = 0.3,
        do_sample = True,
    )

    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    answer   = response.split("assistant")[-1].strip()

    print(f"\nQ: {prompt}")
    print(f"A: {answer}")
    print("-" * 55)

The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Both `max_new_tokens` (=200) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


HEALTHCARE ASSISTANT — MODEL OUTPUT


/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
Both `max_new_tokens` (=200) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Q: What are the early symptoms of diabetes?
A: Early symptoms include frequent urination, thirst, blurred vision, fatigue, and skin rash (hyperpigmentation).
-------------------------------------------------------


Both `max_new_tokens` (=200) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Q: What should I do if someone is having a heart attack?
A: Call emergency services immediately. Check breathing and pulse. Move to safe location. Do CPR if needed.
-------------------------------------------------------


Both `max_new_tokens` (=200) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Q: What are common side effects of ibuprofen?
A: Common side effects include stomach upset, headache, dizziness, and increased bleeding risk in older adults.
-------------------------------------------------------


Both `max_new_tokens` (=200) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Q: How many hours of sleep does an adult need?
A: An adult needs 7-9 hours of quality sleep per night to feel rested and perform well.
-------------------------------------------------------

Q: What are signs of depression I should watch for?
A: Signs include persistent sadness or hopelessness, fatigue, difficulty concentrating, loss of interest in activities, and thoughts of death.
-------------------------------------------------------


### **Before vs After Comparison**

In [15]:
# Load base model to compare
base_model, base_tokenizer = FastLanguageModel.from_pretrained(
    model_name = "Qwen/Qwen2.5-0.5B-Instruct",
    max_seq_length = 1024,
    load_in_4bit = True,
    dtype = None,
)
FastLanguageModel.for_inference(base_model)

test_prompt = "What are the early warning signs of diabetes?"

messages = [
    {
        "role": "system",
        "content": "You are a helpful assistant."
    },
    {
        "role": "user",
        "content": test_prompt
    }
]

# Base model response
base_inputs = base_tokenizer.apply_chat_template(
    messages,
    tokenize = True,
    add_generation_prompt = True,
    return_tensors = "pt"
).to("cuda")

base_outputs = base_model.generate(
    input_ids = base_inputs,
    max_new_tokens = 150,
    temperature = 0.3,
    do_sample = True,
)

base_response = base_tokenizer.decode(
    base_outputs[0],
    skip_special_tokens=True
).split("assistant")[-1].strip()

# Fine-tuned model response
ft_inputs = tokenizer.apply_chat_template(
    messages,
    tokenize = True,
    add_generation_prompt = True,
    return_tensors = "pt"
).to("cuda")

ft_outputs = model.generate(
    input_ids = ft_inputs,
    max_new_tokens = 150,
    temperature = 0.3,
    do_sample = True,
)

ft_response = tokenizer.decode(
    ft_outputs[0],
    skip_special_tokens=True
).split("assistant")[-1].strip()

print("=" * 55)
print("BEFORE vs AFTER FINE-TUNING")
print("=" * 55)
print(f"\nQUESTION: {test_prompt}")
print()
print("BASE MODEL (before):")
print("-" * 30)
print(base_response)
print()
print("FINE-TUNED MODEL (after):")
print("-" * 30)
print(ft_response)


# Free base model from memory
del base_model
torch.cuda.empty_cache()

==((====))==  Unsloth 2026.5.2: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

unsloth/qwen2.5-0.5b-instruct-unsloth-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.


Both `max_new_tokens` (=150) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=150) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


BEFORE vs AFTER FINE-TUNING

QUESTION: What are the early warning signs of diabetes?

BASE MODEL (before):
------------------------------
There are several early warning signs that may indicate the presence of diabetes, including:

1. Increased thirst and frequent urination: This is one of the most common symptoms of diabetes.

2. Frequent urination: This can be caused by high blood sugar levels, which can lead to dehydration.

3. Fatigue: People with diabetes often feel tired or weak, even when they are physically active.

4. Numbness in feet: Diabetes can cause nerve damage, leading to numbness and pain in the feet.

5. Changes in appetite: People with diabetes may experience changes in their appetite, such as increased hunger or decreased appetite.

6. Unusual weight loss: If you lose more weight than expected without trying, it could be a sign

FINE-TUNED MODEL (after):
------------------------------
Early warning signs include frequent urination, excessive thirst, blurred vision, 

### **Push to Hugging Face Hub**

In [ ]:
# You need a Hugging Face account and token
# Get token from: https://huggingface.co/settings/tokens
# Create token with WRITE permission

HF_TOKEN    = ""          # paste your token
HF_USERNAME = ""       # your HF username
MODEL_REPO  = "healthcare-qwen2.5-0.5B-dora"

from huggingface_hub import login
login(token=HF_TOKEN)

# Push adapter to Hub
model.push_to_hub(
    f"{HF_USERNAME}/{MODEL_REPO}",
    token = HF_TOKEN
)

tokenizer.push_to_hub(
    f"{HF_USERNAME}/{MODEL_REPO}",
    token = HF_TOKEN
)

print(f"Model pushed to:")
print(f"https://huggingface.co/{HF_USERNAME}/{MODEL_REPO}")
print()
print("Copy this link for your LinkedIn post and GitHub README")

README.md:   0%|          | 0.00/589 [00:00<?, ?B/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...adapter_model.safetensors: 100%|##########| 8.89MB / 8.89MB            

Saved model to https://huggingface.co/Dhanushkumaramk/healthcare-qwen2.5-0.5B-dora


...

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...mpb53jcu82/tokenizer.json: 100%|##########| 11.4MB / 11.4MB            

No files have been modified since last commit. Skipping to prevent empty commit.


Model pushed to:
https://huggingface.co/Dhanushkumaramk/healthcare-qwen2.5-0.5B-dora

Copy this link for your LinkedIn post and GitHub README
